## Phase 2: Concept Drift Detectors
### Step 1 — Data Initialization & Multivariate Structure Validation

Loads preprocessed IHSG data, validates datetime index, builds drift scoreboard
DataFrames (one per algorithm variant) with NaN for unevaluable initial rows.

#### 1. Configuration & Imports

Defines `Config` dataclass holding all tunable parameters:
- Data paths, feature names, batch/stream algorithm lists
- Window sizes (60, 120), stream warm-up period (60 rows)
- Consensus threshold (30%) for voting in later steps

In [ ]:
from dataclasses import dataclass, field
from pathlib import Path

import numpy as np
import pandas as pd


@dataclass
class Config:
    data_path: str = "../dataset/processed/jkse_preprocessed.csv"
    output_dir: str = "../dataset/processed"
    features: list = field(default_factory=lambda: [
        "Log_Return", "Vol_20d", "Vol_60d", "EMA_5",
        "BB_Middle", "BB_Upper", "BB_Lower",
        "Momentum_5d", "Momentum_20d",
    ])
    batch_windows: list = field(default_factory=lambda: [60, 120])
    batch_algorithms: list = field(default_factory=lambda: [
        "minps", "ks", "psi", "wasserstein",
    ])
    stream_algorithms: list = field(default_factory=lambda: [
        "adwin", "kswin", "ph",
    ])
    stream_warmup: int = 60
    consensus_threshold: float = 0.3

#### 2. Data Loading & Validation

Loads preprocessed JKSE CSV, parses DatetimeIndex, sorts chronologically, and runs sanity assertions:
- Index is unique and datetime-typed
- No remaining NaN values after preprocessing
- Prints date range and shape

In [ ]:
cfg = Config()

print("Config:")
for k, v in cfg.__dict__.items():
    val = str(v)
    if len(val) > 80:
        val = val[:77] + "..."
    print(f"  {k:20s} = {val}")

df = pd.read_csv(cfg.data_path, index_col=0, parse_dates=True)
df = df.sort_index()

assert isinstance(df.index, pd.DatetimeIndex), "Index must be DatetimeIndex"
assert df.index.is_unique, "Index contains duplicate dates"
assert df.isnull().sum().sum() == 0, "Data contains NaN values"

print(f"\nLoaded {df.shape[0]} rows x {df.shape[1]} columns")
print(f"Index type: {df.index.dtype}")
print(f"Date range: {df.index.min().date()}  to  {df.index.max().date()}")

#### 3. Scoreboard Creation

Creates one scoreboard per algorithm-variant (11 total):
- **Batch** (`minps`, `ks`, `psi`, `wasserstein` × 60, 120): `NaN` for first `2*W` rows (insufficient data for two adjacent windows)
- **Stream** (`adwin`, `kswin`, `ph`): `NaN` for first 60 warm-up rows
- All remaining rows initialized to `0.0` (no drift detected yet)

In [ ]:
def create_scoreboards(df: pd.DataFrame, cfg: Config) -> dict[str, pd.DataFrame]:
    scoreboards: dict[str, pd.DataFrame] = {}

    for alg in cfg.batch_algorithms:
        for w in cfg.batch_windows:
            key = f"{alg}_{w}"
            sb = pd.DataFrame(np.nan, index=df.index, columns=cfg.features)
            sb.iloc[2 * w:] = 0.0
            scoreboards[key] = sb

    for alg in cfg.stream_algorithms:
        sb = pd.DataFrame(np.nan, index=df.index, columns=cfg.features)
        sb.iloc[cfg.stream_warmup:] = 0.0
        scoreboards[alg] = sb

    return scoreboards


def scoreboard_summary(scoreboards: dict[str, pd.DataFrame], cfg: Config) -> str:
    lines = [f"Scoreboards created: {len(scoreboards)}"]
    for key, sb in scoreboards.items():
        nan_rows = int(sb.isnull().all(axis=1).sum())
        lines.append(f"  {key:20s}  shape={str(sb.shape):12s}  NaN rows={nan_rows:>4d}")
    return "\n".join(lines)


scoreboards = create_scoreboards(df, cfg)
print("=" * 55)
print("STEP 1 - Result")
print("=" * 55)
print(f"df.shape:              {df.shape}")
print(f"Feature columns:       {cfg.features}")
print(f"Batch windows (days):  {cfg.batch_windows}")
print(f"Stream algorithms:     {cfg.stream_algorithms}")
print(f"Stream warm-up rows:   {cfg.stream_warmup}")
print()
print(scoreboard_summary(scoreboards, cfg))

#### 4. Feature Diagnostics

Exploratory analysis of the 9 feature columns:
- Summary statistics via `describe().T`
- Correlation matrix with heatmap coloring (`RdBu_r`)
- Memory footprint and missing-value check

In [ ]:
print("Feature summary statistics")
print("-" * 55)
display(df[cfg.features].describe().T)

print("\nFeature correlation matrix")
print("-" * 55)
corr = df[cfg.features].corr()
display(corr.style.background_gradient(cmap="RdBu_r", vmin=-1, vmax=1))

missing = int(df[cfg.features].isnull().sum().sum())
print(f"\nMissing values in feature set: {missing}")
print(f"DataFrame memory usage: {df[cfg.features].memory_usage(deep=True).sum() / 1024:.1f} KB")

#### 5. Persist Scoreboards

Saves all 11 scoreboard DataFrames as CSV to `../dataset/processed/`.
Files are named `scoreboard_{key}.csv` (e.g. `scoreboard_minps_60.csv`).
These will be loaded by later steps to write actual drift scores into.

In [ ]:
import os

os.makedirs(cfg.output_dir, exist_ok=True)

for key, sb in scoreboards.items():
    path = Path(cfg.output_dir) / f"scoreboard_{key}.csv"
    sb.to_csv(path)

print(f"Saved {len(scoreboards)} scoreboards to {cfg.output_dir}/")
for key in scoreboards:
    p = Path(cfg.output_dir) / f"scoreboard_{key}.csv"
    size_kb = p.stat().st_size / 1024
    print(f"  {key:20s}  {size_kb:>8.1f} KB")

#### 6. Scoreboard Previews

Displays `head()` of each scoreboard to visually confirm:
- NaN warm-up rows for batch (120 or 240) and stream (60) algorithms
- Correct column structure (9 features)
- Zeroes after the warm-up period

In [ ]:
print("=" * 55)
print("SCOREBOARD PREVIEWS")
print("=" * 55)
for key, sb in scoreboards.items():
    nan_rows = int(sb.isnull().all(axis=1).sum())
    print(f"\n{key}  (NaN rows: {nan_rows})")
    display(sb.head())